# 10 Alignment Strategies


Chạy sau notebook 14 để dùng checkpoint fine-tuned.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
!pip -q install datasets==3.6.0 transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
ds=load_dataset('eriktks/conll2003', trust_remote_code=True)
label_names=ds['test'].features['ner_tags'].feature.names
model_dir=f'{DRIVE_DIR}/bert-base-cased-conll2003'
tokenizer=AutoTokenizer.from_pretrained(model_dir, use_fast=True)
model=AutoModelForTokenClassification.from_pretrained(model_dir)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval()
row=ds['test'][0]
enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256)
word_ids=enc.word_ids()
with torch.no_grad():
    batch={k:torch.tensor([v]).to(device) for k,v in enc.items()}
    logits=model(**batch).logits[0].cpu()

# A: only the first subword receives the word label.
labels_a=[]; previous=None
for wid in word_ids:
    labels_a.append(-100 if wid is None or wid==previous else row['ner_tags'][wid])
    previous=wid

# B: propagate labels; a continuation of B-X becomes I-X.
labels_b=[]; previous=None
for wid in word_ids:
    if wid is None: labels_b.append(-100)
    else:
        tag=label_names[row['ner_tags'][wid]]
        if wid==previous and tag.startswith('B-'): tag='I-'+tag[2:]
        labels_b.append(tag)
    previous=wid

# C: average subword logits to make one prediction per original word.
word_logits=[]
for wid in range(len(row['tokens'])):
    positions=[j for j,x in enumerate(word_ids) if x==wid]
    word_logits.append(logits[positions].mean(0))
pred_c=[label_names[i] for i in torch.stack(word_logits).argmax(-1).tolist()]
print('words:',row['tokens'])
print('tokens:',tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('word_ids:',word_ids)
print('strategy_a_labels:',labels_a)
print('strategy_b_labels:',labels_b)
print('strategy_c_word_predictions:',pred_c)
print('logits:',logits.tolist())


Mounted at /content/drive
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/NLP/lab09/bert-base-cased-conll2003'. Use `repo_type` argument if needed.